<a href="https://colab.research.google.com/github/anaisaoviedo-upb/Energia-2026/blob/main/Regresion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Predicción de tipo Regresion

1. Preparación de Datos
2. División de los datos
3. Aprendizaje del Modelo
4. Evaluación del Modelo: MSE, RMSE, MAE, MAPE
5. Guardamos el modelo

In [ ]:
#Importamos librerías básicas
import pandas as pd # manipulacion dataframes
import numpy as np  # matrices y vectores
import matplotlib.pyplot as plt #gráfica

# 1. Preparación de Datos


In [ ]:
#Cargamos los datos
data = pd.read_excel("datos_limpios.xlsx", sheet_name=0)
data.head()

In [ ]:
#Conocemos los datos
data.info()

In [ ]:
#Corrección tipos de datos
data['Region']=data['Region'].astype('category')
data['Nivel transicion energetica_t+1']=data['Nivel transicion energetica_t+1'].astype('category')
data.info()

In [ ]:
#Eliminamos variables irrelevantes para el modelo predictivo

data = data.drop(columns='Unnamed: 0') # Id de fila irrelevante
data = data.drop(columns='Nivel transicion energetica_t+1') # Se desea predecir como número
data = data.drop(columns='Year') # Year no representa una característica energética del país sino una referencia temporal
data.head()

In [ ]:
#Descripción de variables numéricas
data.describe()

In [ ]:
data.plot(kind='box')

In [ ]:
#Descripción variables categóricas
data['Region'].value_counts().plot(kind='bar')

# **Transformaciones**

In [ ]:
#Dummies para las variables predictoras categóricas
data = pd.get_dummies(data, columns=['Region'], drop_first=False, dtype=int)

data.head()


# 2. División 70-30


In [ ]:
#División 70-30
from sklearn.model_selection import train_test_split
X = data.drop("Energías Renovables_t+1", axis = 1) # Variables predictoras
Y = data['Energías Renovables_t+1'] #Variable objetivo
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, stratify=None)
Y_train.plot(kind='box')

# Aprendizaje  70% y Evaluación 30%

In [ ]:
#Dataframe para comparar los resultados
medidas= pd.DataFrame(index=['mse','rmse','mae','mape','max'])

# Arbol de Regresión
-No se normaliza

In [ ]:
#Creación del modelo con el conjunto de entrenamiento
from sklearn.tree import DecisionTreeRegressor
model_Tree = DecisionTreeRegressor(criterion='squared_error', min_samples_leaf=2, max_depth=20)
model_Tree.fit(X_train, Y_train)#70% entrenamiento



In [ ]:
#Graficar el árbol
from sklearn.tree import plot_tree
nombres_variables=X_train.columns.values
plt.figure(figsize=(20,20))
plot_tree(model_Tree, feature_names=nombres_variables, filled=True,fontsize=8)
plt.show()

In [ ]:
#Evaluación del árbol 30%
from sklearn import metrics
Y_pred = model_Tree.predict(X_test) #30%

#Medidas de evaluación en regresión
mse = metrics.mean_squared_error(Y_test,Y_pred) #No es intepretable
rmse = np.sqrt(mse) #Si es intepretable
mae= metrics.mean_absolute_error(Y_test,Y_pred) #Si es intepretable
#mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred) #Verificar si la objetivo es 0
mape='' #No se puede calcular
max=metrics.max_error(Y_test,Y_pred)
medidas['Arbol']=[mse, rmse, mae, mape,max]
medidas

In [ ]:
#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Tree')
plt.show() # Mostrar la grafica luego de que ya se definio todos los elementos

# Random Forest para Regresión
-No se normaliza

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor

model_rf= RandomForestRegressor(n_estimators=100,  max_samples=0.9, criterion='squared_error',
                              max_depth=20, min_samples_leaf=2)
model_rf.fit(X_train, Y_train) #70%

In [ ]:
importances = model_rf.feature_importances_
feature_names = X_train.columns
forest_importances = pd.Series(importances, index=feature_names)

fig, ax = plt.subplots()
forest_importances.sort_values(ascending=False).plot.bar(ax=ax)
ax.set_title("Importancia de las características (Random Forest)")
ax.set_ylabel("Importancia Media según el criterion")
fig.tight_layout()
plt.show()

In [ ]:
#Evaluación de Random
from sklearn import metrics

Y_pred = model_rf.predict(X_test) #30%

#Medidas de error
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
#mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
mape='' #No se puede calcular
max=metrics.max_error(Y_test,Y_pred)
medidas['RF']=[mse, rmse, mae, mape,max]
print(medidas)

#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Random')
plt.show()

# KNN para Regresión
* Normalización de las var numéricas

In [ ]:
#Normalizacion de las variables numéricas (las dummies no se normalizan)
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()

variables_numericas=['Petróleo_t','Gas Natural_t','Carbón_t','Energía Hidroeléctrica_t','Energías Renovables_t']

#Ajuste de los parametros sobre 100% de los datos (data): max - min
min_max_scaler.fit(data[variables_numericas])

#Se aplica la normalización a 70%  y 30%
X_train[variables_numericas]= min_max_scaler.transform(X_train[variables_numericas]) #70%
X_test[variables_numericas]= min_max_scaler.transform(X_test[variables_numericas]) #30%
X_train.head()

In [ ]:
from sklearn.neighbors import  KNeighborsRegressor
model_Knn = KNeighborsRegressor(n_neighbors=1, metric='euclidean') #minkowski
model_Knn.fit(X_train, Y_train) #70%

In [ ]:
#Evaluación de KNN
from sklearn import metrics

Y_pred = model_Knn.predict(X_test) #30%

#Medidas de error
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
#mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
mape='' #No se puede calcular
max=metrics.max_error(Y_test,Y_pred)
medidas['Knn']=[mse, rmse, mae, mape,max]
print(medidas)

#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Knn')
plt.show()

# 4. Guardamos el modelo seleccionado


In [ ]:
medidas

**Selecion del modelo:**
1. Calidad de los modelos
2. Complejidad computacional: Tree mas liviano computacionalmente

**Selección del modelo:**

In [ ]:
modelo_final=model_Knn

Se entrena modelo final con 100% de los datos (X,Y)

In [ ]:
# Si selecciona Knn -> se debe normalzar X (100% de los datos)
# Si selecciona Tree o RF no se normaliza y se comenta la siguiente línea
X[variables_numericas]= min_max_scaler.transform(X[variables_numericas])

In [ ]:
#Entrenamos modelo final
modelo_final.fit(X, Y) #100%

In [ ]:
import pickle
filename = 'modelo-reg.pkl'
variables=X.columns._values
pickle.dump([modelo_final, min_max_scaler,variables], open(filename, 'wb'))